# Predicting NBA Success Based on Rookie Stats

This notebook represents my workflow for getting data from database in usable format for modeling. There will be another notebook for modeling workflow and feature selection.

Author: Rasheed Martin

In [3]:
import sys
sys.path.append('/Users/rashe/PycharmProjects/PredictingNBACareers')
from utils.duckdb import get_duckdb_connection
import duckdb
import polars as pl

## Let's first get the data for players

### Get NBA Players First Three Season. (Context, we only have data from 2000-2025 so we may have players who started before that but we are treating their first years in the 2000s as their first year)

I will also be able to get their third year stats. So we will do rookie stats, aggregated first three years and third year stats.

In [6]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_season_counts AS (
    SELECT
        player_id,
        COUNT(DISTINCT season_id) AS season_count
    FROM player_stat_values
    GROUP BY player_id
    HAVING COUNT(DISTINCT season_id) > 3
),
first_three_seasons AS (
    SELECT
        psv.player_id,
        s.label,
        ROW_NUMBER() OVER (
            PARTITION BY psv.player_id
            ORDER BY s.label
        ) AS season_rank
    FROM player_stat_values psv
    JOIN player_season_counts psc
      ON psv.player_id = psc.player_id
     JOIN seasons s on psv.season_id = s.season_id
    GROUP BY psv.player_id, s.label
),
selected_stats AS (
    SELECT
        psv.*
    FROM player_stat_values psv
    JOIN seasons s on psv.season_id = s.season_id
    JOIN first_three_seasons fts
      ON psv.player_id = fts.player_id
     AND s.label = fts.label
    WHERE fts.season_rank <= 3
)
SELECT 
    p.name AS Player,
    s.label AS Season,
    t.abbreviation AS Team,
    st.name AS Stat_Type,
    ps.name AS Stat_Name,
    selected_stats.value AS Stat_Value
FROM selected_stats
JOIN players p ON selected_stats.player_id = p.player_id
JOIN seasons s ON selected_stats.season_id = s.season_id
JOIN teams t ON selected_stats.team_id = t.team_id
JOIN stat_types st ON selected_stats.stat_type_id = st.stat_type_id
JOIN stats ps ON selected_stats.stat_id = ps.stat_id
WHERE st.stat_type_id in (2, 3)
ORDER BY p.name, s.label, stat_name;
        """
    ).arrow()
    df_first_three_years = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [7]:
df_first_three_years

Player,Season,Team,Stat_Type,Stat_Name,Stat_Value
str,i32,str,str,str,f32
"""AJ. Price""",2009,"""IND""","""Total_Player_Data""","""2P""",85.0
"""AJ. Price""",2009,"""IND""","""Total_Player_Data""","""2P%""",0.472
"""AJ. Price""",2009,"""IND""","""Total_Player_Data""","""2PA""",180.0
"""AJ. Price""",2009,"""IND""","""Total_Player_Data""","""3P""",60.0
"""AJ. Price""",2009,"""IND""","""Total_Player_Data""","""3P%""",0.345
…,…,…,…,…,…
"""Zydrunas Ilgauskas""",2002,"""CLE""","""Advanced_Data""","""USG%""",27.799999
"""Zydrunas Ilgauskas""",2002,"""CLE""","""Advanced_Data""","""VORP""",1.0
"""Zydrunas Ilgauskas""",2002,"""CLE""","""Advanced_Data""","""WS""",5.0


In [10]:
df_first_three_years = df_first_three_years.pivot(
    index=["Player", "Season"],
    on="Stat_Name",
    values="Stat_Value",
    aggregate_function="first"
)
df_first_three_years

Player,Season,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,Player_Age,STL,STL%,TOV,TOV%,TRB,TRB%,TS%,Trp-Dbl,USG%,VORP,WS,WS/48,eFG%
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AJ. Price""",2009,85.0,0.472,180.0,60.0,0.345,174.0,0.492,106.0,20.5,3.0,0.2,-0.5,-0.4,76.0,9.7,0.8,145.0,0.41,354.0,60.0,0.8,75.0,0.212,56.0,2.0,865.0,-0.1,12.0,1.5,0.4,14.0,53.0,410.0,23.0,35.0,2.0,59.0,13.2,88.0,5.6,0.53,0.0,22.700001,0.3,1.2,0.065,0.494
"""AJ. Price""",2010,73.0,0.427,171.0,41.0,0.275,149.0,0.466,111.0,22.700001,1.0,0.1,-2.9,-1.1,56.0,7.8,0.7,114.0,0.356,320.0,54.0,0.667,81.0,0.253,50.0,0.0,795.0,-1.9,16.0,2.3,-0.4,10.7,61.0,323.0,24.0,29.0,1.8,53.0,13.0,72.0,5.0,0.454,0.0,22.700001,-0.2,0.3,0.02,0.42
"""AJ. Price""",2011,33.0,0.384,86.0,26.0,0.295,88.0,0.506,86.0,23.9,2.0,0.3,-1.0,-0.5,48.0,9.4,0.5,59.0,0.339,174.0,28.0,0.8,35.0,0.201,44.0,1.0,568.0,-0.6,13.0,2.6,0.2,11.5,30.0,172.0,25.0,20.0,1.9,32.0,14.5,61.0,6.0,0.454,0.0,17.700001,0.1,0.7,0.063,0.414
"""Aaron Brooks""",2007,57.0,0.491,116.0,36.0,0.33,109.0,0.484,87.0,23.5,5.0,0.6,-0.2,0.2,43.0,7.9,0.8,93.0,0.413,225.0,42.0,0.857,49.0,0.218,51.0,0.0,608.0,-0.5,13.0,2.5,0.6,13.1,69.0,264.0,23.0,13.0,1.1,44.0,15.1,56.0,5.2,0.535,0.0,21.799999,0.3,1.4,0.112,0.493
"""Aaron Brooks""",2008,203.0,0.428,474.0,113.0,0.366,309.0,0.395,238.0,20.200001,8.0,0.3,-0.9,-0.6,124.0,7.0,2.2,316.0,0.404,783.0,149.0,0.866,172.0,0.22,80.0,35.0,1998.0,-0.3,33.0,2.0,1.4,12.9,152.0,894.0,24.0,46.0,1.2,125.0,12.7,157.0,4.6,0.521,0.0,22.9,0.6,3.6,0.086,0.476
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zion Williamson""",2020,624.0,0.622,1003.0,10.0,0.294,34.0,0.033,226.0,19.700001,39.0,1.8,5.8,-0.3,274.0,14.7,1.6,634.0,0.611,1037.0,369.0,0.698,529.0,0.51,61.0,61.0,2026.0,6.1,167.0,8.9,7.1,27.1,135.0,1647.0,20.0,57.0,1.3,167.0,11.6,441.0,11.8,0.649,0.0,29.799999,4.0,8.7,0.205,0.616
"""Zion Williamson""",2022,278.0,0.618,450.0,7.0,0.368,19.0,0.041,133.0,24.4,16.0,1.6,5.8,1.0,144.0,17.0,1.2,285.0,0.608,469.0,177.0,0.714,248.0,0.529,29.0,29.0,956.0,4.8,58.0,6.9,2.3,25.200001,65.0,754.0,22.0,32.0,1.6,99.0,14.6,202.0,12.0,0.652,0.0,30.4,1.9,3.6,0.18,0.615
"""Zydrunas Ilgauskas""",2000,114.0,0.491,232.0,0.0,0.0,2.0,0.009,18.0,5.4,37.0,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616.0,-1.4,65.0,12.3,0.1,16.0,78.0,281.0,25.0,15.0,1.3,60.0,18.299999,160.0,15.0,0.524,0.0,24.1,0.0,0.7,0.056,0.487


Going to filter out rookies and other players who did not play 3 or more years

In [11]:
players_who_played_first_three_years = df_first_three_years.select("Player").unique().to_series().to_list()

Below we calculate the percent growth from year 1 to year 2 and year 2 to year 3 and then the avg annual year over year growth. The resulting data frame will have their rookie stats included

In [14]:
import polars as pl

# List your stat columns
stat_cols = df_first_three_years.select(pl.exclude(["Player", "Season", "Player_Age"])).columns

# Build list of aggregate expressions
exprs = []

for stat in stat_cols:
    col = pl.col(stat)
    exprs.extend([
        # Year‑1→2 growth
        ((col.shift(-1).over("Player") - col) / col.shift(-1).over("Player") * 100)
            .alias(f"{stat}_growth_y1_y2"),
        # Year‑2→3 growth
        ((col.shift(-2).over("Player") - col.shift(-1).over("Player"))
         / col.shift(-2).over("Player") * 100)
            .alias(f"{stat}_growth_y2_y3"),
        # Average of the two
        ((( (col.shift(-1).over("Player") - col) / col.shift(-1).over("Player") * 100 )
          + ( (col.shift(-2).over("Player") - col.shift(-1).over("Player")) / col.shift(-2).over("Player") * 100 )
         ) / 2).alias(f"{stat}_avg_yoy_growth")
    ])

# Apply to only first-year rows per player
df_pct_diff_three_years = (
    df_first_three_years.select(["Player", "Season", "Player_Age"] + stat_cols)
    .sort(["Player", "Season"])
    .with_columns(exprs)
    .filter(pl.col("Season") == pl.col("Season").min().over("Player"))  # keep only the year‑1 row
)

df_pct_diff_three_years


Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,STL_avg_yoy_growth,STL%_growth_y1_y2,STL%_growth_y2_y3,STL%_avg_yoy_growth,TOV_growth_y1_y2,TOV_growth_y2_y3,TOV_avg_yoy_growth,TOV%_growth_y1_y2,TOV%_growth_y2_y3,TOV%_avg_yoy_growth,TRB_growth_y1_y2,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AJ. Price""",2009,23.0,85.0,0.472,180.0,60.0,0.345,174.0,0.492,106.0,20.5,3.0,0.2,-0.5,-0.4,76.0,9.7,0.8,145.0,0.41,354.0,60.0,0.8,75.0,0.212,56.0,2.0,865.0,-0.1,12.0,1.5,0.4,14.0,53.0,410.0,35.0,…,-32.844826,-11.111115,5.263159,-2.923978,-11.320755,-65.625,-38.472878,-1.53846,10.344828,4.403184,-22.222223,-18.032785,-20.127504,-11.999998,16.666668,2.333335,-16.740082,0.0,-8.370041,NaN,NaN,NaN,0.0,-28.248587,-14.124293,250.0,300.0,275.0,-300.0,57.142853,-121.428574,-225.0,68.253967,-78.373016,-17.619047,-1.449271,-9.534159
"""Aaron Brooks""",2007,23.0,57.0,0.491,116.0,36.0,0.33,109.0,0.484,87.0,23.5,5.0,0.6,-0.2,0.2,43.0,7.9,0.8,93.0,0.413,225.0,42.0,0.857,49.0,0.218,51.0,0.0,608.0,-0.5,13.0,2.5,0.6,13.1,69.0,264.0,13.0,…,52.536232,8.333335,0.0,4.166667,64.800003,46.120689,55.460346,-18.897642,7.299271,-5.799186,64.331207,26.976744,45.653976,-13.043477,-9.523812,-11.283645,-2.68714,5.100181,1.206521,NaN,NaN,NaN,4.803495,10.894946,7.84922,50.0,68.421051,59.210526,61.111103,34.545456,47.828278,-30.232557,5.4945,-12.369028,-3.571426,6.849309,1.638941
"""Aaron Gordon""",2014,19.0,80.0,0.5,160.0,13.0,0.271,48.0,0.231,33.0,6.3,22.0,2.2,-2.7,-0.1,123.0,17.799999,0.7,93.0,0.447,208.0,44.0,0.721,61.0,0.293,47.0,8.0,797.0,-2.6,46.0,6.4,0.3,11.4,83.0,243.0,21.0,…,36.818775,18.750004,-14.285718,2.232143,42.424244,25.842697,34.133469,-54.444439,-5.882353,-30.163395,66.666672,-25.185186,20.740742,20.529804,-57.291664,-18.38093,4.436227,-2.075479,1.180374,NaN,NaN,NaN,10.40462,13.930353,12.167486,114.285721,-133.333328,-9.523804,81.481483,-45.945946,17.767769,56.83453,-80.519478,-11.842474,6.090377,-2.004006,2.043185
"""Aaron Gray""",2007,23.0,103.0,0.507,203.0,0.0,0.0,1.0,0.005,41.0,11.4,16.0,2.0,-3.5,0.0,98.0,18.700001,0.8,103.0,0.505,204.0,56.0,0.566,99.0,0.485,61.0,1.0,613.0,-3.6,70.0,12.5,0.1,13.9,117.0,262.0,21.0,…,-52.777779,-79.999992,33.333336,-23.333328,-42.5,-122.222221,-82.361115,-8.72093,-7.500005,-8.110468,23.636364,-107.547165,-41.955399,11.93182,11.111106,11.521463,-4.133852,7.80399,1.835069,NaN,NaN,NaN,-51.724136,11.585363,-20.069386,-0.0,300.0,150.0,38.461536,-44.444443,-2.991453,23.255817,39.436619,31.346218,-4.123707,5.273439,0.574866
"""Aaron Holiday""",2018,22.0,62.0,0.459,135.0,43.0,0.339,127.0,0.485,87.0,19.299999,13.0,1.8,-1.6,0.1,62.0,10.4,0.8,105.0,0.401,262.0,41.0,0.82,50.0,0.191,50.0,0.0,646.0,-1.7,5.0,0.9,0.1,11.9,71.0,294.0,21.0,…,21.126484,5.882354,5.555551,5.718952,54.545456,-33.333336,10.60606,3.90625,-4.065041,-0.079395,57.051285,-75.280899,-9.114807,-9.433962,-29.268301,-19.351131,0.57582,-3.57853,-1.501355,NaN,NaN,NaN,-17.112293,4.10256,-6.504867,50.0,128.571426,89.285713,57.142853,-949.999878,-446.428528,-3.174598,-600.000061,-301.587341,1.629325,-5.139184,-1.754929
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

### Getting general players roy info, all rookie team info, draft pick, draft position

In [16]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
SELECT 
    p.name AS Player,
    p.draft_pick AS Draft_Pick,
    p.draft_class AS Draft_Class,
    r.rank AS ROY_Rank,
    r.first_place_votes AS ROY_Votes,
    r.point_share AS ROY_Point_Share,
    a.rank AS All_Rookie_Rank,
    a.first_place_votes AS All_Rookie_First_Place_Votes,
    a.second_place_votes AS All_Rookie_Second_Place_Votes,
    a.points_share AS All_Rookie_Point_Share

FROM nba_data.main.players p
LEFT JOIN nba_data.main.roy_voting r on p.player_id = r.player_id 
LEFT JOIN nba_data.main.all_rookie_voting a on p.player_id = a.player_id
        """
    ).arrow()
    df_players_rookie_awards = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [17]:
df_players_rookie_awards

Player,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share
str,i32,i32,i32,i32,f32,str,i32,i32,f32
"""Nikola Jokic""",41,2014,3,0,0.091,"""1ST Team""",73,40,0.715
"""Kemba Walker""",9,2011,11,0,0.005,"""Received Votes""",0,7,0.121
"""Andrew Bogut""",1,2005,3,0,0.157,"""1ST Team""",26,3,0.948
"""Marc Gasol""",48,2007,8,0,0.007,"""2ND Team""",2,21,0.431
"""Jaylen Brown""",3,2016,8,0,0.002,"""2ND Team""",24,58,0.53
…,…,…,…,…,…,…,…,…,…
"""Walter Herrmann""",null,null,10,0,0.008,"""2ND Team""",1,8,0.172
"""Jorge Garbajosa""",null,null,8,0,0.027,"""1ST Team""",13,11,0.638
"""Juan Carlos Navarro""",40,2002,6,0,0.005,"""2ND Team""",5,14,0.414


Let's merge their rookie awards with the pct differential and rookie stats

In [19]:
df_pct_diff_rookie_stats = df_pct_diff_three_years.join(
    df_players_rookie_awards,
    on="Player",
    how="left"
)
df_pct_diff_rookie_stats

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,TOV%_avg_yoy_growth,TRB_growth_y1_y2,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,str,i32,i32,f32
"""AJ. Price""",2009,23.0,85.0,0.472,180.0,60.0,0.345,174.0,0.492,106.0,20.5,3.0,0.2,-0.5,-0.4,76.0,9.7,0.8,145.0,0.41,354.0,60.0,0.8,75.0,0.212,56.0,2.0,865.0,-0.1,12.0,1.5,0.4,14.0,53.0,410.0,35.0,…,4.403184,-22.222223,-18.032785,-20.127504,-11.999998,16.666668,2.333335,-16.740082,0.0,-8.370041,NaN,NaN,NaN,0.0,-28.248587,-14.124293,250.0,300.0,275.0,-300.0,57.142853,-121.428574,-225.0,68.253967,-78.373016,-17.619047,-1.449271,-9.534159,null,null,null,null,null,null,null,null,null
"""Aaron Brooks""",2007,23.0,57.0,0.491,116.0,36.0,0.33,109.0,0.484,87.0,23.5,5.0,0.6,-0.2,0.2,43.0,7.9,0.8,93.0,0.413,225.0,42.0,0.857,49.0,0.218,51.0,0.0,608.0,-0.5,13.0,2.5,0.6,13.1,69.0,264.0,13.0,…,-5.799186,64.331207,26.976744,45.653976,-13.043477,-9.523812,-11.283645,-2.68714,5.100181,1.206521,NaN,NaN,NaN,4.803495,10.894946,7.84922,50.0,68.421051,59.210526,61.111103,34.545456,47.828278,-30.232557,5.4945,-12.369028,-3.571426,6.849309,1.638941,26,2007,null,null,null,"""Received Votes""",0,1,0.017
"""Aaron Gordon""",2014,19.0,80.0,0.5,160.0,13.0,0.271,48.0,0.231,33.0,6.3,22.0,2.2,-2.7,-0.1,123.0,17.799999,0.7,93.0,0.447,208.0,44.0,0.721,61.0,0.293,47.0,8.0,797.0,-2.6,46.0,6.4,0.3,11.4,83.0,243.0,21.0,…,-30.163395,66.666672,-25.185186,20.740742,20.529804,-57.291664,-18.38093,4.436227,-2.075479,1.180374,NaN,NaN,NaN,10.40462,13.930353,12.167486,114.285721,-133.333328,-9.523804,81.481483,-45.945946,17.767769,56.83453,-80.519478,-11.842474,6.090377,-2.004006,2.043185,4,2014,null,null,null,"""Received Votes""",1,3,0.019
"""Aaron Gray""",2007,23.0,103.0,0.507,203.0,0.0,0.0,1.0,0.005,41.0,11.4,16.0,2.0,-3.5,0.0,98.0,18.700001,0.8,103.0,0.505,204.0,56.0,0.566,99.0,0.485,61.0,1.0,613.0,-3.6,70.0,12.5,0.1,13.9,117.0,262.0,21.0,…,-8.110468,23.636364,-107.547165,-41.955399,11.93182,11.111106,11.521463,-4.133852,7.80399,1.835069,NaN,NaN,NaN,-51.724136,11.585363,-20.069386,-0.0,300.0,150.0,38.461536,-44.444443,-2.991453,23.255817,39.436619,31.346218,-4.123707,5.273439,0.574866,49,2007,null,null,null,null,null,null,null
"""Aaron Holiday""",2018,22.0,62.0,0.459,135.0,43.0,0.339,127.0,0.485,87.0,19.299999,13.0,1.8,-1.6,0.1,62.0,10.4,0.8,105.0,0.401,262.0,41.0,0.82,50.0,0.191,50.0,0.0,646.0,-1.7,5.0,0.9,0.1,11.9,71.0,294.0,21.0,…,-0.079395,57.051285,-75.280899,-9.114807,-9.433962,-29.268301,-19.351131,0.57582,-3.57853,-1.501355,NaN,NaN,NaN,-17.112293,4.10256,-6.504867,50.0,128.571426,89.285713,57.142853,-949.999878,-446.428528,-3.174598,-600.000061,-301.587341,1.629325,-5.139184,-1.754929,23,2018,null,null,null,"""Received Votes""",0,1,0.005
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zeljko Rebraca""",2001,29.0,189.0,0.505,374.0,0.0,null,0.0,0.0,38.0,5.7,73.0,4.4,-2.3,0.6,206.0,20.5,1.9,189.0,0.505,374.0,135.0,0.771,175.0,0.468,74.0,4.0,11

Do some feature creation

### Getting team data to see if they made the playoffs prior

In [20]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_seasons AS (
    SELECT DISTINCT
        psv.player_id,
        psv.team_id,
        psv.season_id
    FROM player_stat_values psv
)

SELECT 
    p.name AS player_name,
    s.label AS season,
    t.name AS team,
    CASE 
        WHEN tsi_prev.made_playoffs IS NULL THEN NULL
        ELSE tsi_prev.made_playoffs
    END AS team_made_playoffs_last_year
FROM player_seasons ps
JOIN players p ON p.player_id = ps.player_id
JOIN teams t ON t.team_id = ps.team_id
JOIN seasons s ON s.season_id = ps.season_id
JOIN seasons s_prev ON s_prev.label = s.label - 1
LEFT JOIN team_season_info tsi_prev 
    ON tsi_prev.team_id = ps.team_id 
    AND tsi_prev.season_id = s_prev.season_id
        """
    ).arrow()
    df_team_playoffs_prior = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [22]:
df_with_team_player_stats = df_pct_diff_rookie_stats.join(df_team_playoffs_prior, left_on=["Player", "Season"], right_on=["player_name", "season"], how="left")
df_with_team_player_stats

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share,team,team_made_playoffs_last_year
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,str,i32,i32,f32,str,bool
"""AJ. Price""",2009,23.0,85.0,0.472,180.0,60.0,0.345,174.0,0.492,106.0,20.5,3.0,0.2,-0.5,-0.4,76.0,9.7,0.8,145.0,0.41,354.0,60.0,0.8,75.0,0.212,56.0,2.0,865.0,-0.1,12.0,1.5,0.4,14.0,53.0,410.0,35.0,…,-18.032785,-20.127504,-11.999998,16.666668,2.333335,-16.740082,0.0,-8.370041,NaN,NaN,NaN,0.0,-28.248587,-14.124293,250.0,300.0,275.0,-300.0,57.142853,-121.428574,-225.0,68.253967,-78.373016,-17.619047,-1.449271,-9.534159,null,null,null,null,null,null,null,null,null,"""Indiana Pacers""",false
"""Aaron Brooks""",2007,23.0,57.0,0.491,116.0,36.0,0.33,109.0,0.484,87.0,23.5,5.0,0.6,-0.2,0.2,43.0,7.9,0.8,93.0,0.413,225.0,42.0,0.857,49.0,0.218,51.0,0.0,608.0,-0.5,13.0,2.5,0.6,13.1,69.0,264.0,13.0,…,26.976744,45.653976,-13.043477,-9.523812,-11.283645,-2.68714,5.100181,1.206521,NaN,NaN,NaN,4.803495,10.894946,7.84922,50.0,68.421051,59.210526,61.111103,34.545456,47.828278,-30.232557,5.4945,-12.369028,-3.571426,6.849309,1.638941,26,2007,null,null,null,"""Received Votes""",0,1,0.017,"""Houston Rockets""",true
"""Aaron Gordon""",2014,19.0,80.0,0.5,160.0,13.0,0.271,48.0,0.231,33.0,6.3,22.0,2.2,-2.7,-0.1,123.0,17.799999,0.7,93.0,0.447,208.0,44.0,0.721,61.0,0.293,47.0,8.0,797.0,-2.6,46.0,6.4,0.3,11.4,83.0,243.0,21.0,…,-25.185186,20.740742,20.529804,-57.291664,-18.38093,4.436227,-2.075479,1.180374,NaN,NaN,NaN,10.40462,13.930353,12.167486,114.285721,-133.333328,-9.523804,81.481483,-45.945946,17.767769,56.83453,-80.519478,-11.842474,6.090377,-2.004006,2.043185,4,2014,null,null,null,"""Received Votes""",1,3,0.019,"""Orlando Magic""",false
"""Aaron Gray""",2007,23.0,103.0,0.507,203.0,0.0,0.0,1.0,0.005,41.0,11.4,16.0,2.0,-3.5,0.0,98.0,18.700001,0.8,103.0,0.505,204.0,56.0,0.566,99.0,0.485,61.0,1.0,613.0,-3.6,70.0,12.5,0.1,13.9,117.0,262.0,21.0,…,-107.547165,-41.955399,11.93182,11.111106,11.521463,-4.133852,7.80399,1.835069,NaN,NaN,NaN,-51.724136,11.585363,-20.069386,-0.0,300.0,150.0,38.461536,-44.444443,-2.991453,23.255817,39.436619,31.346218,-4.123707,5.273439,0.574866,49,2007,null,null,null,null,null,null,null,"""Chicago Bulls""",true
"""Aaron Holiday""",2018,22.0,62.0,0.459,135.0,43.0,0.339,127.0,0.485,87.0,19.299999,13.0,1.8,-1.6,0.1,62.0,10.4,0.8,105.0,0.401,262.0,41.0,0.82,50.0,0.191,50.0,0.0,646.0,-1.7,5.0,0.9,0.1,11.9,71.0,294.0,21.0,…,-75.280899,-9.114807,-9.433962,-29.268301,-19.351131,0.57582,-3.57853,-1.501355,NaN,NaN,NaN,-17.112293,4.10256,-6.504867,50.0,128.571426,89.285713,57.142853,-949.999878,-446.428528,-3.174598,-600.000061,-301.587341,1.629325,-5.139184,-1.754929,23,2018,null,null,null,"""Received Votes""",0,1,0.005,"""Indiana Pacers""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zeljko Rebraca""",2001,29.0,189.0,0.505,374.0,0.0,null,0.0,0.0,38.0,5.7,73.0,4.4,-2.3,0.6,206.0,20.5,1.9,189.0,0.505,374.0,135.0,0

### Get third year stats

In [34]:
df_three_year_stats = df_first_three_years.join(
    df_first_three_years.group_by("Player").agg(pl.col("Season").max().alias("Season")),
    on=["Player", "Season"]
)
df_three_year_stats

Player,Season,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,Player_Age,STL,STL%,TOV,TOV%,TRB,TRB%,TS%,Trp-Dbl,USG%,VORP,WS,WS/48,eFG%
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AJ. Price""",2011,33.0,0.384,86.0,26.0,0.295,88.0,0.506,86.0,23.9,2.0,0.3,-1.0,-0.5,48.0,9.4,0.5,59.0,0.339,174.0,28.0,0.8,35.0,0.201,44.0,1.0,568.0,-0.6,13.0,2.6,0.2,11.5,30.0,172.0,25.0,20.0,1.9,32.0,14.5,61.0,6.0,0.454,0.0,17.700001,0.1,0.7,0.063,0.414
"""Aaron Brooks""",2009,366.0,0.454,806.0,209.0,0.398,525.0,0.394,434.0,25.6,14.0,0.3,0.7,-1.6,161.0,6.6,1.6,575.0,0.432,1331.0,245.0,0.822,298.0,0.224,82.0,82.0,2919.0,2.3,54.0,2.0,3.9,16.0,199.0,1604.0,25.0,69.0,1.2,232.0,13.7,215.0,4.2,0.549,0.0,25.700001,1.9,5.5,0.091,0.511
"""Aaron Gordon""",2016,316.0,0.528,598.0,77.0,0.288,267.0,0.309,150.0,10.5,40.0,1.4,-1.0,-0.7,289.0,14.1,1.7,393.0,0.454,865.0,156.0,0.719,217.0,0.251,80.0,72.0,2298.0,-0.3,116.0,5.3,2.0,14.5,172.0,1019.0,21.0,65.0,1.4,89.0,8.5,405.0,9.6,0.53,0.0,20.1,0.6,3.7,0.077,0.499
"""Aaron Gray""",2009,42.0,0.512,82.0,0.0,null,0.0,0.0,22.0,10.6,11.0,2.7,-1.3,-0.2,61.0,23.1,0.4,42.0,0.512,82.0,20.0,0.714,28.0,0.341,32.0,0.0,311.0,-1.1,45.0,16.700001,0.5,16.6,54.0,104.0,25.0,9.0,1.5,18.0,16.0,106.0,19.799999,0.551,0.0,16.4,0.1,0.9,0.142,0.512
"""Aaron Holiday""",2020,103.0,0.406,254.0,67.0,0.368,182.0,0.417,123.0,14.0,13.0,0.9,-4.2,-0.9,74.0,6.8,0.8,170.0,0.39,436.0,68.0,0.819,83.0,0.19,66.0,8.0,1176.0,-3.3,15.0,1.4,-0.6,9.2,94.0,475.0,24.0,46.0,1.8,66.0,12.3,89.0,4.1,0.503,0.0,19.5,-0.7,0.2,0.009,0.467
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zeljko Rebraca""",2003,34.0,0.442,77.0,0.0,null,0.0,0.0,6.0,4.0,11.0,3.0,-3.9,0.6,35.0,14.6,0.4,34.0,0.442,77.0,23.0,0.767,30.0,0.39,24.0,2.0,273.0,-4.5,23.0,9.9,0.1,10.5,52.0,91.0,31.0,5.0,1.0,17.0,15.9,58.0,12.2,0.504,0.0,18.200001,-0.1,0.5,0.097,0.442
"""Zendon Hamilton""",2002,2.0,0.4,5.0,0.0,null,0.0,0.0,0.0,0.0,0.0,0.0,-3.1,0.0,3.0,29.700001,0.0,2.0,0.4,5.0,2.0,1.0,2.0,0.4,3.0,0.0,12.0,-3.1,1.0,9.1,0.0,17.700001,2.0,6.0,27.0,1.0,4.5,1.0,14.5,4.0,19.0,0.51,0.0,26.200001,0.0,0.0,0.107,0.4
"""Ziaire Williams""",2023,92.0,0.487,189.0,58.0,0.307,189.0,0.5,75.0,11.1,9.0,0.8,-4.5,-0.4,145.0,15.9,1.1,150.0,0.397,378.0,62.0,0.827,75.0,0.198,51.0,15.0,1038.0,-4.0,35.0,3.5,-0.9,9.3,85.0,420.0,22.0,36.0,1.7,66.0,13.8,180.0,9.5,0.511,0.0,19.700001,-0.6,0.2,0.01,0.474


In [39]:
df_without_target = df_with_team_player_stats.join(df_three_year_stats.rename({col: f"{col}_third" for col in df_first_three_years.columns if col not in  ["Player"]}), on=["Player"], how="left")
df_without_target

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,BLK%_third,BPM_third,DBPM_third,DRB_third,DRB%_third,DWS_third,FG_third,FG%_third,FGA_third,FT_third,FT%_third,FTA_third,FTr_third,G_third,GS_third,MP_third,OBPM_third,ORB_third,ORB%_third,OWS_third,PER_third,PF_third,PTS_third,Player_Age_third,STL_third,STL%_third,TOV_third,TOV%_third,TRB_third,TRB%_third,TS%_third,Trp-Dbl_third,USG%_third,VORP_third,WS_third,WS/48_third,eFG%_third
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AJ. Price""",2009,23.0,85.0,0.472,180.0,60.0,0.345,174.0,0.492,106.0,20.5,3.0,0.2,-0.5,-0.4,76.0,9.7,0.8,145.0,0.41,354.0,60.0,0.8,75.0,0.212,56.0,2.0,865.0,-0.1,12.0,1.5,0.4,14.0,53.0,410.0,35.0,…,0.3,-1.0,-0.5,48.0,9.4,0.5,59.0,0.339,174.0,28.0,0.8,35.0,0.201,44.0,1.0,568.0,-0.6,13.0,2.6,0.2,11.5,30.0,172.0,25.0,20.0,1.9,32.0,14.5,61.0,6.0,0.454,0.0,17.700001,0.1,0.7,0.063,0.414
"""Aaron Brooks""",2007,23.0,57.0,0.491,116.0,36.0,0.33,109.0,0.484,87.0,23.5,5.0,0.6,-0.2,0.2,43.0,7.9,0.8,93.0,0.413,225.0,42.0,0.857,49.0,0.218,51.0,0.0,608.0,-0.5,13.0,2.5,0.6,13.1,69.0,264.0,13.0,…,0.3,0.7,-1.6,161.0,6.6,1.6,575.0,0.432,1331.0,245.0,0.822,298.0,0.224,82.0,82.0,2919.0,2.3,54.0,2.0,3.9,16.0,199.0,1604.0,25.0,69.0,1.2,232.0,13.7,215.0,4.2,0.549,0.0,25.700001,1.9,5.5,0.091,0.511
"""Aaron Gordon""",2014,19.0,80.0,0.5,160.0,13.0,0.271,48.0,0.231,33.0,6.3,22.0,2.2,-2.7,-0.1,123.0,17.799999,0.7,93.0,0.447,208.0,44.0,0.721,61.0,0.293,47.0,8.0,797.0,-2.6,46.0,6.4,0.3,11.4,83.0,243.0,21.0,…,1.4,-1.0,-0.7,289.0,14.1,1.7,393.0,0.454,865.0,156.0,0.719,217.0,0.251,80.0,72.0,2298.0,-0.3,116.0,5.3,2.0,14.5,172.0,1019.0,21.0,65.0,1.4,89.0,8.5,405.0,9.6,0.53,0.0,20.1,0.6,3.7,0.077,0.499
"""Aaron Gray""",2007,23.0,103.0,0.507,203.0,0.0,0.0,1.0,0.005,41.0,11.4,16.0,2.0,-3.5,0.0,98.0,18.700001,0.8,103.0,0.505,204.0,56.0,0.566,99.0,0.485,61.0,1.0,613.0,-3.6,70.0,12.5,0.1,13.9,117.0,262.0,21.0,…,2.7,-1.3,-0.2,61.0,23.1,0.4,42.0,0.512,82.0,20.0,0.714,28.0,0.341,32.0,0.0,311.0,-1.1,45.0,16.700001,0.5,16.6,54.0,104.0,25.0,9.0,1.5,18.0,16.0,106.0,19.799999,0.551,0.0,16.4,0.1,0.9,0.142,0.512
"""Aaron Holiday""",2018,22.0,62.0,0.459,135.0,43.0,0.339,127.0,0.485,87.0,19.299999,13.0,1.8,-1.6,0.1,62.0,10.4,0.8,105.0,0.401,262.0,41.0,0.82,50.0,0.191,50.0,0.0,646.0,-1.7,5.0,0.9,0.1,11.9,71.0,294.0,21.0,…,0.9,-4.2,-0.9,74.0,6.8,0.8,170.0,0.39,436.0,68.0,0.819,83.0,0.19,66.0,8.0,1176.0,-3.3,15.0,1.4,-0.6,9.2,94.0,475.0,24.0,46.0,1.8,66.0,12.3,89.0,4.1,0.503,0.0,19.5,-0.7,0.2,0.009,0.467
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zeljko Rebraca""",2001,29.0,189.0,0.505,374.0,0.0,null,0.0,0.0,38.0,5.7,73.0,4.4,-2.3,0.6,206.0,20.5,1.9,189.0,0.505,374.0,135.0,0.771,175.0,0.468,74.0,4.0,1179.0,-2.9,84.0,8.4,1.4,16.200001,192.0,513.0,28.0,…,3.0,-3.9,0.6,35.0,14.6,0.4,34.0,0.442,77.0,23.0,0.767,30.0,0.39,24.0,2.0,273.0,-4.5,23.0,9.9,0.1,10.5,52.0,91.0,31.0,5.0,1.0,17.0,15.9,58.0,12.2,0.504,0.0,18.200001,-0.1,0.5,0.097,0.442
"""Zendon Hamilton""",2000,25.0,2.0,0.222,9.0,0.0,null,0.0,0.0,0.0,0.0,0.0,0.0,-15.0,-6.1,5.0,29.299999,0.0,2.0,0.222,9.0,5.0,0.625,8.0,0.889,3.0,0.0,19.0,-8.9,3.0,18.6,-0.1,5.7,4.0,9.0,0.0,…,0.0,-3.1,0.0,3.0,29.700001,0.0,2.0,0.4,5.0,2.0,1.0,2.0,0.4,3.0,0.0,12.0,-3.1,1.0,9.1,0.0,17.700001,2.0,6.0,27.0,1.0,4.5,1.0,14.5,4.0,19.0,0.51,0.0,26.200001,0.0,0.0,0.107,0.4
"""Ziaire Williams""",2021,20.0,115.0,0.632,182.0,76.0,0.314,242.0,0.571,65.0,6.3,12.0,0.8,-3.0,-0.6,103.0,8.2,1.1,191.0,0.45,424.0,43.0,0.782,55.0,0.13,62.0,31.0,1346.0,-2.4,26.0,2.0,1.1,9.7,113.0,501.0,35.0,…,0.8,-4.5,-0.4,145.0,15.9,1.1,150.0,0.397,3

Looking at the null count, it is obvious based on context that if the players have null values at any step they did not take any or have any. 

From the above analysis, we can see that the following pairs of features have the highest absolute correlation:
Field Goals and Points
2 Points Attempted vs 2 Points Made
Minutes Per Games

In [41]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_stats as (
SELECT
    MAX(CASE WHEN st.name = 'WS' THEN psv.value ELSE NULL END) AS ws,
    MAX(CASE WHEN st.name = 'USG%' THEN psv.value ELSE NULL END) AS usage,
	MAX(CASE WHEN st.name = 'G' THEN psv.value ELSE NULL END) AS games_played,
	MAX(CASE WHEN st.name = 'MP' THEN psv.value ELSE NULL END) AS minutes_played,
	MAX(CASE WHEN st.name = 'GS' THEN psv.value ELSE NULL END) AS games_started,
	MAX(CASE WHEN st.name = 'BPM' THEN psv.value ELSE NULL END) AS box_plus_minus,
	psv.season_id,
	psv.player_id,
	psv.team_id
FROM
	nba_data.main.player_stat_values psv
JOIN nba_data.main.stats st on
	st.stat_id = psv.stat_id
JOIN nba_data.main.stat_types s on s.stat_type_id = psv.stat_type_id
WHERE psv.stat_type_id in (3, 2) -- Only Total and Advanced Stats
GROUP BY psv.player_id, psv.season_id, psv.team_id
)
SELECT
 p.name as Player,
 ps.ws as Win_Share,
 ps."usage" as Usage,
 ps.games_played as Games_Played,
 ps.minutes_played as Minutes_Played,
 ps.games_started as Games_Started,
 ps.box_plus_minus as Box_Plus_Minus,
 a.all_nba_team as All_Nba_Team,
 a.allstar as AllStar,
 a.six_moy_rank as Six_MOY,
 s."label" as Season,
 t.name As Team
 
FROM nba_data.main.award_info a
JOIN nba_data.main.players p on a.player_id = p.player_id
JOIN player_stats ps on ps.player_id = p.player_id AND ps.season_id = a.season_id
JOIN nba_data.main.seasons s on s.season_id = ps.season_id
JOIN nba_data.main.teams t on t.team_id = a.team_id
        """
    ).arrow()
    df_award_info = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [42]:
df_award_info

Player,Win_Share,Usage,Games_Played,Minutes_Played,Games_Started,Box_Plus_Minus,All_Nba_Team,AllStar,Six_MOY,Season,Team
str,f32,f32,f32,f32,f32,f32,i32,bool,i32,i32,str
"""Dale Davis""",8.5,13.7,78.0,2447.0,77.0,1.1,0,false,0,2001,"""Portland Trail Blazers"""
"""Juwan Howard""",4.9,22.299999,81.0,2635.0,72.0,-1.3,0,false,0,2001,"""Unknown"""
"""Trajan Langdon""",0.6,22.700001,44.0,477.0,0.0,-1.3,0,false,0,2001,"""Cleveland Cavaliers"""
"""Ernest Brown""",-0.2,24.5,3.0,21.0,0.0,-18.200001,0,false,0,2001,"""Miami Heat"""
"""Shaquille O'Neal""",13.2,31.799999,67.0,2422.0,66.0,8.0,1,true,0,2001,"""Los Angeles Lakers"""
…,…,…,…,…,…,…,…,…,…,…,…
"""Xavier Tillman Sr""",1.4,14.8,54.0,974.0,15.0,-1.2,0,false,0,2023,"""Unknown"""
"""Thaddeus Young""",1.3,12.6,33.0,439.0,6.0,0.3,0,false,0,2023,"""Toronto Raptors"""
"""Marques Bolden""",0.3,10.9,11.0,121.0,2.0,-2.7,0,false,0,2023,"""Milwaukee Bucks"""


In [43]:
df_award_info.null_count()

Player,Win_Share,Usage,Games_Played,Minutes_Played,Games_Started,Box_Plus_Minus,All_Nba_Team,AllStar,Six_MOY,Season,Team
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,3,0,0,0,3,0,0,0,0,0


In [44]:
def min_max_norm(col: str) -> pl.Expr:
    return ((pl.col(col) - pl.col(col).min()) / (pl.col(col).max() - pl.col(col).min())).alias(f"{col}_norm")

Creating the target variable to model against

In [45]:
# Aggregate stats per player
player_summary = df_award_info.group_by("Player").agg([
    pl.col("All_Nba_Team").max().alias("All_Nba_Team_Max"),
    pl.col("AllStar").max().alias("AllStar_Max"),
    pl.col("Games_Played").mean().alias("Games_Played_Mean"),
    pl.col("Win_Share").sum().alias("Win_Share_Sum"),
    pl.col("Box_Plus_Minus").mean().alias("Box_Plus_Minus_Mean"),
    pl.col("Minutes_Played").mean().alias("Minutes_Played_Mean"),
    pl.col("Games_Started").mean().alias("Games_Started_Mean"),
    pl.col("Usage").mean().alias("Usage_Mean")
])
player_summary = player_summary.with_columns([
    (
        0.5 * min_max_norm("Win_Share_Sum") +
        0.4 * min_max_norm("Box_Plus_Minus_Mean") +
        0.4 * min_max_norm("Usage_Mean") +
        0.2 * min_max_norm("Games_Started_Mean") +
        0.1 * min_max_norm("Minutes_Played_Mean")).alias("Player_Score")
])
# Assign player type
player_summary = player_summary.with_columns([
    pl.when(pl.col("All_Nba_Team_Max") > 0)
      .then(pl.lit("All NBA Caliber"))
      .when(pl.col("AllStar_Max") > 0)
      .then(pl.lit("All Star Caliber"))
      .when(
          pl.col("Player_Score") > 0.4
      ).then(pl.lit("Star Player"))
      .when(
          pl.col("Player_Score") > 0.3
      ).then(pl.lit("Role Player"))
      .otherwise(pl.lit("Bust Player"))
      .alias("Player_Type")
])

player_summary

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Mickael Gelabale""",0,false,48.333332,3.0,-1.966667,782.666687,9.0,14.066666,0.289572,"""Bust Player"""
"""Aaron Holiday""",0,false,64.0,10.099998,-1.828571,1052.285767,9.428572,17.957144,0.334637,"""Role Player"""
"""Nic Claxton""",0,false,51.833332,26.099998,0.816667,1337.333374,38.166668,15.283333,0.43709,"""Star Player"""
"""Malik Monk""",0,false,65.375,15.1,-1.3625,1492.875,10.375,24.074997,0.39622,"""Role Player"""
"""Alex Sarr""",0,false,67.0,-0.3,-3.3,1814.0,67.0,23.5,0.507877,"""Star Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Jordan Walsh""",0,false,30.5,0.6,-3.85,243.0,1.0,10.7,0.22298,"""Bust Player"""
"""Sam Perkins""",0,false,64.0,2.0,1.0,999.0,41.0,12.2,0.371998,"""Role Player"""
"""Clifford Robinson""",0,false,75.14286,20.9,-1.142857,2254.857178,50.857143,18.757143,0.502734,"""Star Player"""


In [46]:
from pathlib import Path
path = Path.cwd().parent.parent / "data/clean_data" / "player_label.csv"
player_summary.select(["Player", "Player_Type"]).write_csv(path)

In [47]:
player_labels_only_three_years = player_summary.join(pl.DataFrame({"Player": players_who_played_first_three_years}), on="Player", how="inner")

In [48]:
path = Path.cwd().parent.parent / "data/clean_data" / "player_label_only_three_years.csv"
player_labels_only_three_years.select(["Player", "Player_Type"]).write_csv(path)

Combine this player_label with our full dataset

In [51]:
path = Path.cwd().parent.parent / "data/refined_raw_data" / "Player_Type_Modeling"
path.mkdir(parents=True, exist_ok=True)
df_without_target.join(
    player_labels_only_three_years.rename({"Player_Type": "Player_Label"}),
    on="Player",
    how="left"
).write_csv(
    path / "df_without_target.csv"
)